# Thémines - consolidation des chroniques

Deux sondes : **CTD** (Diver autonome : niveau, conductivité, température), **TROLL** (Aqua TROLL : conductivité, température, turbidité, O2, chlorophylle).

Même logique qu'à Cabouy, la station de référence : une colonne par sonde, une
sonde choisie automatiquement à chaque pas dans l'ordre `ORDRE`, des périodes
imposées à la main quand le graphe montre que ce choix n'est pas le bon, et un
recalage mesuré à chaque changement de sonde.

## 1. Imports

In [ ]:
import os
import re
import unicodedata
from io import StringIO

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go

## 2. Chemins d'accès

In [ ]:
BASE        = r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Thémines\Gaetan"
CTD_PATH    = os.path.join(BASE, r"Données brutes\CTD")
VUSITU_PATH = os.path.join(BASE, r"Données brutes\TROLL")
BARO_PATH   = r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\1 - Données BARO\Gourdon baro\Patm Calès et Thémines.xlsx"
PLUIE_PATH  = r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Saint Sauveur\Gaetan\Données brutes\Pluie_BV_Ouysse.csv"

OLDDATA_PATH     = os.path.join(BASE, "Themine_consolide_OLD.csv")
UTC_CTD_PATH     = os.path.join(BASE, r"UTC_CTD.xlsx")
UTC_TROLL_PATH   = os.path.join(BASE, "UTC_Troll.xlsx")
PUNCTUAL_NIVEAU  = os.path.join(BASE, "punctual_measurements_Niveau.xlsx")
PUNCTUAL_CONDUCT = os.path.join(BASE, "punctual_measurements_Brute.xlsx")
SORTIE_CONSOLIDE = os.path.join(BASE, "Themines_consolide.xlsx")
SORTIE_FINALE    = os.path.join(BASE, "Themines_final.xlsx")
SORTIE_SVG       = os.path.join(BASE, "Graphes.svg")

PREFIXE_CTD = "Thémines"
BARO_COL    = "Patm Ouysse Calès [hPa]"
PAS         = "1h"

## 3. Fonctions de lecture

Les pièges de format : en-tête Diver à une ligne variable et pied `END OF DATA`,
virgules décimales, conductivité en mS/cm ou µS/cm selon la campagne, encodages
mélangés. La table UTC nomme les fichiers exactement, sinon la campagne est ignorée.

In [ ]:
def _sans_accents(t):
    d = unicodedata.normalize("NFKD", str(t))
    return "".join(c for c in d if not unicodedata.combining(c)).lower()


def _lire_lignes(chemin):
    for enc in ("utf-8-sig", "utf-8", "cp1252", "latin1"):
        try:
            with open(chemin, "r", encoding=enc) as f:
                return f.read().splitlines(), enc
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Aucun encodage ne convient pour {chemin}")


def _en_datetime(serie):
    """Garde le format qui convertit le plus de lignes."""
    txt = serie.astype("string").str.strip()
    meilleur, n_ok = None, -1
    for fmt in ("%Y/%m/%d %H:%M:%S", "%Y-%m-%d %H:%M:%S", "%d/%m/%Y %H:%M:%S", "%d/%m/%Y %H:%M"):
        e = pd.to_datetime(txt, format=fmt, errors="coerce")
        if e.notna().sum() > n_ok:
            meilleur, n_ok = e, e.notna().sum()
    if n_ok < len(txt):                      # format inconnu : lecture libre
        libre = pd.to_datetime(txt, errors="coerce", dayfirst=True)
        meilleur = libre if libre.notna().sum() > n_ok else meilleur
    return meilleur


def fichiers(path, motif):
    """Fichiers du dossier contenant `motif`, triés par numéro."""
    noms = [f for f in os.listdir(path)
            if motif.lower() in f.lower() and f.lower().endswith((".csv", ".txt", ".mon"))]
    return sorted(noms, key=lambda n: (int(re.findall(r"\d+", n)[0]) if re.findall(r"\d+", n) else 10 ** 9, n))


def lire_CTD(nom, path=CTD_PATH):
    """Export Diver : en-tête cherchée par contenu, pied END OF DATA reconnu,
    conductivité convertie d'après l'unité entre crochets."""
    chemin = os.path.join(path, nom)
    lignes, encodage = _lire_lignes(chemin)
    entete = next((i for i, l in enumerate(lignes[:200])
                   if _sans_accents(l).lstrip("\ufeff").startswith("date/time")), None)
    if entete is None:
        raise ValueError(f"En-tête 'Date/time' introuvable dans {nom}")

    df = pd.read_csv(chemin, sep=";", encoding=encodage, skiprows=entete, dtype=str, engine="python")
    df.columns = [c.strip() for c in df.columns]
    col = df.columns[0]
    brut = df[col].astype("string")
    fin = brut.map(lambda v: pd.notna(v) and "end of data" in _sans_accents(v)).fillna(False)
    df = df.loc[~(fin | brut.isna() | (brut.str.strip() == ""))].copy()

    df["Date/time"] = _en_datetime(df[col]).dt.round(PAS)
    for c in df.columns:
        if c not in ("Date/time", col):
            df[c] = pd.to_numeric(df[c].astype("string").str.strip()
                                  .str.replace(",", ".", regex=False), errors="coerce")
    for c in list(df.columns):
        if "cond" in _sans_accents(c):
            u = re.search(r"\[([^\]]*)\]", c)
            df[c] = df[c] * (1000.0 if u and _sans_accents(u.group(1)).startswith("ms/cm") else 1.0)
            df = df.rename(columns={c: "Cond_(µS/cm)"})
            break
    return df.loc[df["Date/time"].notna()].sort_values("Date/time")


#: Libellés VuSitu (sans numéro de série, sans accent) vers les noms du projet.
NOMS_TROLL = {
    "conductivite specifique (us/cm)":        "Cond_Troll_(µS/cm)",
    "temperature (c)":                        "température_Troll_(°C)",
    "turbidite (ntu)":                        "Turbidity_Troll_(NTU)",
    "concentration rdo (mg/l)":               "O2_Troll_(mg/l)",
    "saturation rdo (%sat)":                  "O2 (%Sat)",
    "fluorescence de chlorophylle-a (rfu)":   "FluorescenceChloro_a_Troll_(RFU)",
    "concentration de chlorophylle-a (ug/l)": "ConcentrationChloro_a_(µg/l)",
}


def lire_VuSitu(nom, path=VUSITU_PATH):
    """Export VuSitu : guillemets retirés, numéro de série retiré par regex."""
    lignes, _ = _lire_lignes(os.path.join(path, nom))
    df = pd.read_csv(StringIO("\n".join(l.replace('"', "") for l in lignes)), sep=",")
    cle = lambda c: (_sans_accents(re.sub(r"\s*\(\d{4,}\)\s*$", "", str(c)).strip())
                     .replace("\u03bc", "u").replace("\u00b5", "u").replace("\u00b0", ""))
    col = next((c for c in df.columns if "date" in _sans_accents(c)), df.columns[0])
    df["DATE"] = _en_datetime(df[col]).dt.round(PAS)
    return df.drop(columns=[col]).rename(
        columns={c: NOMS_TROLL[cle(c)] for c in df.columns if cle(c) in NOMS_TROLL})


def en_utc(df, nom, metadata, col_date="Date/time", col_utc="UTC Fichier"):
    """Ramène les horodatages en UTC. Correspondance EXACTE sur le nom du
    fichier : sinon la campagne est ignoree et le message la nomme."""
    ligne = metadata.loc[metadata["Nom fichier"] == nom, col_utc]
    if ligne.empty:
        raise ValueError(f"'{nom}' absent de la colonne 'Nom fichier'. "
                         f"À corriger dans la table UTC.")
    v = ligne.values[0]
    m = re.search(r"([+-]?\d+(?:[.,]\d+)?)", str(v))
    decalage = float(v) if isinstance(v, (int, float, np.number)) and pd.notna(v) else (
        float(m.group(1).replace(",", ".")) if m else 0.0)
    df = df.copy()
    df[col_date] = df[col_date] - pd.Timedelta(hours=decalage)
    return df, decalage


#: Gamme physique par mot-clé de nom de colonne. Pas de seuil sur le NIVEAU :
#: il n'a pas d'origine absolue tant qu'il n'est pas calé.
GAMMES = {"cond": (30, 5000), "temp": (-2, 30), "turbid": (0, 4000),
          "o2": (0, 25), "chloro": (0, 500)}


def appliquer_gammes(df):
    """Met à NaN ce qui est physiquement impossible, voie par voie."""
    for col in df.columns:
        if not pd.api.types.is_numeric_dtype(df[col]):
            continue
        for cle, (mini, maxi) in GAMMES.items():
            if cle in _sans_accents(col):
                hors = (df[col] < mini) | (df[col] > maxi)
                if hors.any():
                    print(f"  {col:36s} {int(hors.sum()):6d} hors [{mini}, {maxi}]")
                    df.loc[hors, col] = np.nan
                break
    return df

### Fonctions de correction

`decaler` porte le choix du sens : `aval` pour une marche réelle (capteur déplacé),
`amont` pour ramener l'historique sur la référence actuelle, `tout` pour un calage
d'appareil. `raccorder` mesure le décalage **à la jonction**, `choisir_sondes` découpe
la chronique en périodes et `fusionner` les enchaîne en recalant chaque changement.

In [ ]:
def graphe(traces, titre="", ylab="", points=None, col_point=None):
    """`traces` = liste de (série, nom, couleur). Scattergl : une chronique
    horaire pluriannuelle s'affiche sans saturer le navigateur."""
    fig = go.Figure()
    for serie, nom, couleur in traces:
        fig.add_trace(go.Scattergl(x=serie.index, y=serie, mode="lines", name=nom,
                                   line=dict(color=couleur, width=1.3)))
    if points is not None and col_point in points.columns:
        corr = points.get("Correction", pd.Series("Non", index=points.index))
        fig.add_trace(go.Scattergl(
            x=points["Datetime"], y=points[col_point], mode="markers", name="points de contrôle",
            marker=dict(symbol="x", size=10,
                        color=["red" if str(v).strip() == "Oui" else "royalblue" for v in corr])))
    fig.update_layout(title=titre, xaxis_title="Date", yaxis_title=ylab,
                      template="plotly_white", hovermode="x unified")
    fig.show()          

#: Couleur de chaque sonde, la même sur tous les graphes.
COULEURS = {"OTT": "#1f77b4", "TROLL": "#d62728", "CTD": "#2ca02c"}

def decaler(serie, date, valeur, sens="tout"):
    """Ajoute `valeur` a toute la serie, a l'aval de `date` (incluse) ou a
    l'amont (strictement avant)."""
    if sens == "tout":
        return serie + valeur
    date = pd.to_datetime(date)
    m = np.asarray(serie.index >= date if sens == "aval" else serie.index < date)
    return serie.where(~m, serie + valeur)


def ecarter(df, voies_ecartees):
    """Passe à NaN une periode d'une voie : (debut, fin, colonne, motif).
    agit sur la voie brute, avant la fusion : une sonde douteuse est retiree, une autre
    prend le relais si elle mesure.
    """
    for entree in voies_ecartees:
        if len(entree) != 4:
            raise ValueError(f"VOIES_ECARTEES attend (début, fin, colonne, motif), "
                             f"reçu {entree!r}")
        debut, fin, col, motif = entree
        if col not in df.columns:
            raise ValueError(f"VOIES_ECARTEES : colonne inconnue {col!r}")
        debut, fin = sorted([pd.to_datetime(debut), pd.to_datetime(fin)])
        m = np.asarray((df.index >= debut) & (df.index <= fin))
        print(f"  {debut:%d/%m/%Y %H:%M} - {fin:%d/%m/%Y %H:%M}  {col} : "
              f"{int((m & df[col].notna().to_numpy()).sum())} pas écartés ({motif})")
        df[col] = df[col].mask(m)
    return df


def caler(serie, points, col_valeur, tolerance_h=1):
    """Recale la serie sur les mesures ponctuelles marquees Oui (dans le fichier punctual measurements),
    correction toujours vers l'aval. Chaque point est trace, appliqué ou non."""
    for _, l in points.dropna(subset=["Datetime"]).sort_values("Datetime").iterrows():
        date, cible = l["Datetime"], l.get(col_valeur)
        if pd.isna(cible) or str(l.get("Correction", "Non")).strip() != "Oui":
            continue
        mesures = serie.dropna()
        i = mesures.index[np.abs((mesures.index - date).to_numpy()).argmin()] if len(mesures) else None
        if i is None or abs((i - date).total_seconds()) > tolerance_h * 3600:
            print(f"  {date:%d/%m/%Y %H:%M} : ignoré, pas de mesure à moins de {tolerance_h} h")
            continue
        d = float(cible) - float(serie.loc[i])
        print(f"  {date:%d/%m/%Y %H:%M} : {float(serie.loc[i]):.1f} vers {float(cible):.1f}, "
              f"décalage {d:+.2f} appliqué vers l'aval")
        serie = decaler(serie, date, d, "aval")
    return serie


def raccorder(ancienne, nouvelle, transition, sonde, unite="", trou_max_h=12, fenetre_h=24):
    """Decalage a ajouter a `nouvelle` pour qu'elle prolonge `ancienne`.

    Mesure A LA JONCTION : mediane des ecarts sur les `fenetre_h` pas communs
    les plus proches de la transition, de part et d'autre. Une mediane sur des
    semaines melangerait la fin de vie de la sonde qui s'arrete avec son
    comportement normal et laisserait une marche a la jonction, ce qui est
    justement ce qu'on cherche a eviter.

    Sans aucun recouvrement, raccord bout a bout si le trou est court ;
    au-dela de `trou_max_h` rien ne permet de caler les deux sondes l'une sur
    l'autre, donc on ne recale pas et on laisse le trou.
    """
    commun = (ancienne.notna() & nouvelle.notna()).to_numpy()
    if commun.any():
        dates = ancienne.index[commun]
        ordre = np.argsort(np.abs((dates - transition).to_numpy()), kind="stable")
        proches = dates[ordre[:fenetre_h]]
        m = commun & np.asarray(ancienne.index.isin(proches))
        d = float((ancienne[m] - nouvelle[m]).median())
        note = (f"jonction, {int(m.sum())} pas communs du "
                f"{proches.min():%d/%m/%Y %H:%M} au {proches.max():%d/%m/%Y %H:%M}")
    else:
        a = ancienne[ancienne.index <= transition].dropna()
        b = nouvelle[nouvelle.index >= transition].dropna()
        trou = ((b.index[0] - a.index[-1]).total_seconds() / 3600
                if len(a) and len(b) else np.inf)
        if trou <= trou_max_h:
            d = float(a.iloc[-1] - b.iloc[0])
            note = f"trou de {trou:.0f} h, raccord bout à bout"
        else:
            d, note = 0.0, f"trou de {trou:.0f} h : aucun recalage possible"
    print(f"  {sonde:6s} recalée de {d:+.2f} {unite}  ({note})")
    return d


def choisir_sondes(voies, ordre, exceptions=(), duree_mini_h=12):
    """Decoupe la chronique en periodes (debut, fin, sonde).

    A chaque pas, la premiere sonde disponible de `ordre`. Sur une periode
    d'exception (debut, fin, sonde), la sonde nommee passe en tete la ou elle
    mesure. Un bloc plus court que `duree_mini_h` est absorbe par le
    precedent : on ne change pas de sonde pour boucher un trou de quelques
    heures, c'est l'interpolation qui s'en charge.
    """
    index = next(iter(voies.values())).index
    choix = pd.Series(pd.NA, index=index, dtype="object")
    for sonde in [s for s in ordre if s in voies] + [s for s in voies if s not in ordre]:
        choix = choix.where(choix.notna() | voies[sonde].isna(), sonde)
    for entree in exceptions:
        if len(entree) != 3:
            raise ValueError(f"Une période imposée attend (début, fin, sonde), reçu {entree!r}")
        debut, fin, sonde = entree
        if sonde not in voies:
            raise ValueError(f"Période imposée : sonde inconnue {sonde!r} "
                             f"(attendu : {', '.join(voies)})")
        d, f = pd.to_datetime(debut), pd.to_datetime(fin)
        if f < d:
            raise ValueError(f"Période imposée {sonde} : fin ({fin}) antérieure au début ({debut})")
        p = np.asarray((index >= d) & (index <= f))
        choix = choix.where(~(p & voies[sonde].notna().to_numpy()), sonde)

    choix, blocs = choix.dropna(), []
    mini = pd.Timedelta(hours=duree_mini_h)
    for _, g in choix.groupby((choix != choix.shift()).cumsum()):
        if blocs and (g.iloc[0] == blocs[-1][2] or g.index[-1] - g.index[0] < mini):
            blocs[-1][1] = g.index[-1]
        else:
            blocs.append([g.index[0], g.index[-1], g.iloc[0]])
    return [tuple(b) for b in blocs]


def fusionner(voies, periodes, unite="", trou_max_h=12, fenetre_h=24):
    """Chronique d'une grandeur. `voies` = {sonde: serie}.

    Chaque periode n'utilise QUE sa sonde : aucune autre ne vient combler ses
    lacunes. A chaque changement de sonde, la nouvelle est recalee sur la
    precedente A LA JONCTION, en cascade depuis la premiere periode, qui fixe
    le zero.
    """
    index = next(iter(voies.values())).index
    valeur = pd.Series(np.nan, index=index)
    source = pd.Series(pd.NA, index=index, dtype="object")
    decalage, precedente = 0.0, None
    for debut, fin, sonde in periodes:
        transition = pd.to_datetime(debut)
        p = np.asarray((index >= transition) & (index <= pd.to_datetime(fin)))
        serie = voies[sonde]
        print(f"  {transition:%d/%m/%Y %H:%M} - {pd.to_datetime(fin):%d/%m/%Y %H:%M}  {sonde}")
        if precedente is not None and sonde != precedente:
            decalage = raccorder(voies[precedente] + decalage, serie, transition,
                                 sonde, unite, trou_max_h, fenetre_h)
        pris = p & serie.notna().to_numpy()
        valeur[pris], source[pris] = serie[pris] + decalage, sonde
        precedente = sonde
    return valeur, source

## 4. CTD : lecture, UTC et compensation barométrique

In [ ]:
HPA_EN_CMH2O = 1.019716 #`1 hPa = 1.019716 cmH2O`.

metadata = pd.read_excel(UTC_CTD_PATH)
baro = pd.read_excel(BARO_PATH)[["DATE", BARO_COL]]
baro["DATE"] = pd.to_datetime(baro["DATE"], errors="coerce")
baro = baro.dropna(subset=["DATE"]).drop_duplicates("DATE")

morceaux = []
for nom in fichiers(CTD_PATH, PREFIXE_CTD):
    try:
        CTD, decalage = en_utc(lire_CTD(nom), nom, metadata)
    except Exception as e:
        print(f"  IGNORÉ  {nom} : {e}")
        continue
    m = pd.merge(CTD, baro, left_on="Date/time", right_on="DATE", how="left")
    m["Niveau_(cm)"] = m["Pression[cmH2O]"] - m[BARO_COL] * HPA_EN_CMH2O
    m = m.rename(columns={"Température[°C]": "Temp_(°C)"})
    morceaux.append(m[["Date/time", "Niveau_(cm)", "Cond_(µS/cm)", "Temp_(°C)"]])
    print(f"  {nom:45s} UTC+{decalage:g} vers UTC   ({len(CTD)} lignes)")

merge_ctd_df = pd.concat(morceaux, ignore_index=True).sort_values("Date/time", kind="stable")
merge_ctd_df["DATE"] = merge_ctd_df["Date/time"]
print(f"\n{len(morceaux)} campagne(s), {len(merge_ctd_df)} enregistrements en UTC.")

## 5. Raccordement à l'ancienne chronique

L'ancien fichier consolidé et les campagnes récentes sont la **même sonde CTD**, séparées
par un trou d'exploitation : le décalage est mesuré à la jonction et appliqué aux
campagnes, pour que la chronique soit continue.

In [ ]:
#: L'ancien consolidé n'emploie pas toujours les noms de colonnes du notebook.
RENOMMAGE_OLD = {
    "Niveau": "Niveau_CTD_(cm)",
    "Conducti": "Cond_CTD_(µS/cm)",
    "Temp": "Temp _CTD(°C)",
    "Xtroll": "Cond_Troll_(µS/cm)",
    "TempTroll": "température_Troll_(°C)",
    "Turbidity": "Turbidity_Troll_(NTU)",
    "O2": "O2_Troll_(mg/l)",
    "FluoChloro_a": "FluorescenceChloro_a_Troll_(RFU)",
}

olddata_df = pd.read_csv(OLDDATA_PATH, sep=";")
olddata_df["DATE"] = pd.to_datetime(olddata_df["DATE"], errors="coerce", dayfirst=True)
olddata_df = (olddata_df.rename(columns=RENOMMAGE_OLD).dropna(subset=["DATE"])
              .sort_values("DATE"))

ancien, nouveau = olddata_df.set_index("DATE"), merge_ctd_df.set_index("DATE")
grandeurs = [("Niveau", "Niveau_CTD_(cm)", "Niveau_(cm)", "cm"),
             ("Conductivité", "Cond_CTD_(µS/cm)", "Cond_(µS/cm)", "µS/cm")]
for nom, col_a, col_n, unite in grandeurs:
    a, n = ancien[col_a].dropna(), nouveau[col_n].dropna()
    d = float(a.iloc[-1] - n.iloc[0]) if len(a) and len(n) else 0.0
    print(f"{nom:13s} : {d:+9.2f} {unite:6s} mesuré à la jonction  "
          f"{a.index[-1]:%d/%m/%Y %H:%M} vers {n.index[0]:%d/%m/%Y %H:%M}")
    merge_ctd_df[col_n] = merge_ctd_df[col_n] + d

bord = olddata_df["DATE"].max()
fig, axes = plt.subplots(2, 1, figsize=(8, 3), sharex=True)
for ax, (nom, col_a, col_n, unite) in zip(axes, grandeurs):
    ax.plot(olddata_df["DATE"], olddata_df[col_a], color="green", label="ancienne chronique")
    ax.plot(merge_ctd_df["DATE"], merge_ctd_df[col_n], color="blue", label="campagnes raccordées")
    ax.set_ylabel(f"{nom} ({unite})")
    ax.legend(loc="upper left")
axes[0].set_xlim(bord - pd.Timedelta(days=7), bord + pd.Timedelta(days=7))
plt.tight_layout()
plt.show()

## 6. TROLL : exports VuSitu

In [ ]:
metadata_troll = pd.read_excel(UTC_TROLL_PATH, sheet_name=0)

morceaux = []
for nom in fichiers(VUSITU_PATH, "VuSitu"):
    try:
        df_v, decalage = en_utc(lire_VuSitu(nom), nom, metadata_troll, col_date="DATE")
    except Exception as e:
        print(f"  IGNORÉ  {nom} : {e}")
        continue
    morceaux.append(df_v)
    print(f"  {nom:45s} UTC+{decalage:g} vers UTC   ({len(df_v)} lignes)")

merge_troll_df = (pd.concat(morceaux, ignore_index=True).dropna(subset=["DATE"])
                  .sort_values("DATE", kind="stable"))
print(f"\n{len(morceaux)} fichier(s), {len(merge_troll_df)} enregistrements en UTC.")

## 7. Assemblage : une colonne par sonde

In [ ]:
COLONNES_CTD   = ["Niveau_CTD_(cm)", "Cond_CTD_(µS/cm)", "Temp _CTD(°C)"]
COLONNES_TROLL = ["Cond_Troll_(µS/cm)", "température_Troll_(°C)", "Turbidity_Troll_(NTU)",
                  "O2_Troll_(mg/l)", "O2 (%Sat)", "FluorescenceChloro_a_Troll_(RFU)",
                  "ConcentrationChloro_a_(µg/l)"]


def empiler(morceaux, colonnes):
    pile = pd.concat(morceaux, ignore_index=True).dropna(subset=["DATE"])
    pile = pile[["DATE"] + [c for c in colonnes if c in pile.columns]]
    return (pile.sort_values("DATE", kind="stable")
            .drop_duplicates("DATE", keep="first").set_index("DATE"))


piles = [
    empiler([olddata_df,
             merge_ctd_df.rename(columns={"Niveau_(cm)": "Niveau_CTD_(cm)",
                                          "Cond_(µS/cm)": "Cond_CTD_(µS/cm)",
                                          "Temp_(°C)": "Temp _CTD(°C)"})], COLONNES_CTD),
    empiler([olddata_df, merge_troll_df], COLONNES_TROLL),
]

grille = pd.date_range(min(p.index.min() for p in piles),
                       max(p.index.max() for p in piles), freq=PAS, name="DATE")
full_data = pd.DataFrame(index=grille)
for pile in piles:
    for col in pile.columns:
        full_data[col] = pile[col].reindex(grille)
print(f"{len(full_data)} pas horaires, du {grille.min():%d/%m/%Y} au {grille.max():%d/%m/%Y}")

print("Hors gamme physique :")
full_data = appliquer_gammes(full_data)

#: Les sondes de chaque grandeur : {grandeur: {sonde: colonne}}.
SONDES = {
    "Niveau_(cm)":        {"CTD": "Niveau_CTD_(cm)"},
    "Conductivité":       {"TROLL": "Cond_Troll_(µS/cm)", "CTD": "Cond_CTD_(µS/cm)"},
    "Température":        {"TROLL": "température_Troll_(°C)", "CTD": "Temp _CTD(°C)"},
    "Turbidité_(NTU)":    {"TROLL": "Turbidity_Troll_(NTU)"},
    "O2_(mg/l)":          {"TROLL": "O2_Troll_(mg/l)"},
    "Chlorophylle_(RFU)": {"TROLL": "FluorescenceChloro_a_Troll_(RFU)"},
}
PARAMETRES = list(SONDES)

#: Ordre de préférence automatique : à chaque pas, la première sonde qui mesure.
ORDRE = ["CTD", "TROLL"]

BRUT = full_data.copy()
full_data.to_excel(SORTIE_CONSOLIDE)
print(f"\nfichier fusionné : {SORTIE_CONSOLIDE}")

## 8. Corrections capteur

Les deux seules corrections qui portent sur une **sonde**, avant toute fusion.

`VOIES_ECARTEES` met des mesures à l'écart : la voie est retirée avant la fusion, donc
une autre sonde prend le relais si elle mesure ; s'il n'y en a pas, la lacune reste et
l'interpolation ne comblera pas plus de 12 h.

`CALAGES_SONDE` déplace une sonde entière, ou son passé, ou son avenir.

In [ ]:
#: (début, fin, colonne, motif) : mesures mises à l'écart, toutes grandeurs.
VOIES_ECARTEES = [
    ("2024-03-07 11:00", "2024-03-07 11:00", "Niveau_CTD_(cm)", "pic isolé"),
    ("2020-01-21 12:00", "2020-01-21 12:00", "Niveau_CTD_(cm)", "pic isolé"),
    ("2019-02-27 17:00", "2019-03-30 11:00", "Cond_CTD_(µS/cm)", "valeurs aberrantes"),
    ("2024-07-31 11:00", "2024-09-18 16:00", "Cond_CTD_(µS/cm)", "valeurs aberrantes"),
]

#: (date d'ancrage, sonde, grandeur, décalage, sens) : ajustements manuels.
#: sens = "amont" (avant la date) | "aval" (à partir de la date) | "tout".
CALAGES_SONDE = [

]
print("Voies écartées :")
CORRIGE = ecarter(BRUT.copy(), VOIES_ECARTEES)

def voies_calees(grandeur):
    """Les séries des sondes d'une grandeur, écarts et calages appliqués."""
    series = {s: CORRIGE[c] for s, c in SONDES[grandeur].items()}
    for date, sonde, cible, valeur, sens in CALAGES_SONDE:
        if cible == grandeur and sonde in series:
            series[sonde] = decaler(series[sonde], date, valeur, sens)
            print(f"  {sonde} : {valeur:+.2f} en {sens} du "
                  f"{pd.to_datetime(date):%d/%m/%Y %H:%M}")
    return series

## 9. Comparaison des sources

À lancer pour juger quelle sonde garder sur une période, avant d'écrire une période
imposée dans les cellules suivantes.

In [ ]:
PARAMETRE = "Conductivité"   # "Niveau_(cm)", "Conductivité", "Température", "Turbidité_(NTU)", "O2_(mg/l)", "Chlorophylle_(RFU)"

graphe([(BRUT[col], f"sonde {sonde}", COULEURS[sonde])
        for sonde, col in SONDES[PARAMETRE].items()],
       titre=f"{PARAMETRE} : les sondes disponibles", ylab=PARAMETRE)

## 10. Niveau

La sonde est choisie automatiquement, dans l'ordre `ORDRE` déclaré à l'assemblage.
`SONDE_PRIORITAIRE_NIVEAU` sert à imposer une autre sonde sur une période précise. Les
périodes retenues sont affichées, avec le recalage appliqué à chaque changement.

In [ ]:
#: (début, fin, sonde imposée) : sort du choix automatique sur cette période.
SONDE_PRIORITAIRE_NIVEAU = [

]
points_niveau = pd.read_excel(PUNCTUAL_NIVEAU)
points_niveau["Datetime"] = pd.to_datetime(points_niveau["Jour"], dayfirst=True, errors="coerce")

print("Calages de sonde :")
voies = voies_calees("Niveau_(cm)")
print("Périodes retenues :")
avant, source = fusionner(voies, choisir_sondes(voies, ORDRE, SONDE_PRIORITAIRE_NIVEAU), "cm")

print("Points de contrôle :")
niveau = caler(avant, points_niveau, "Hauteur (cm)")
full_data["Niveau_(cm)"], full_data["Niveau_(cm)_source"] = niveau, source
print("Pas de temps par sonde :", source.value_counts().to_dict())

graphe([(serie, f"sonde {sonde}", COULEURS[sonde]) for sonde, serie in voies.items()]
       + [(avant, "fusion, avant correction", "lightgrey"),
          (niveau, "chronique corrigée", "black")],
       titre="Niveau", ylab="Niveau (cm)", points=points_niveau, col_point="Hauteur (cm)")

## 11. Conductivité

In [ ]:
#: (début, fin, sonde imposée) : sort du choix automatique sur cette période.
SONDE_PRIORITAIRE_COND = [

]
points_cond = pd.read_excel(PUNCTUAL_CONDUCT)
points_cond["Datetime"] = pd.to_datetime(points_cond["Jour"], dayfirst=True, errors="coerce")

print("Calages de sonde :")
voies = voies_calees("Conductivité")
print("Périodes retenues :")
avant, source = fusionner(voies, choisir_sondes(voies, ORDRE, SONDE_PRIORITAIRE_COND), "µS/cm")

print("Points de contrôle :")
cond = caler(avant, points_cond, "Conductivité")
print("Pas de temps par sonde :", source.value_counts().to_dict())

graphe([(serie, f"sonde {sonde}", COULEURS[sonde]) for sonde, serie in voies.items()]
       + [(avant, "fusion, avant correction", "lightgrey"),
          (cond, "chronique fusionnée et calée", "black")],
       titre="Conductivité", ylab="Conductivité (µS/cm)",
       points=points_cond, col_point="Conductivité")

### Filtre IQR et lissage

Post-traitement appliqué **après** la fusion et le calage sur les points de contrôle :
c'est cette chronique nettoyée qui alimente `full_data` et le fichier final.

In [ ]:
FENETRE_IQR, K_IQR = "48h", 0.0   # k = 0 : pas de filtre
LISSAGE_H = 0                     # 0 = pas de lissage ; sinon médiane glissante, en heures

hors = pd.Series(False, index=cond.index)
if K_IQR:
    r = cond.rolling(FENETRE_IQR, center=True, min_periods=8)
    q1, q3 = r.quantile(0.25), r.quantile(0.75)
    hors = ((cond < q1 - K_IQR * (q3 - q1)) | (cond > q3 + K_IQR * (q3 - q1))).fillna(False)
cond_iqr = cond.mask(hors)
print(f"Filtre IQR ({FENETRE_IQR}, k={K_IQR}) : {int(hors.sum())} valeurs écartées")

if LISSAGE_H:
    cond_iqr = cond_iqr.rolling(f"{LISSAGE_H}h", center=True, min_periods=1).median()
    print(f"Lissage : médiane glissante sur {LISSAGE_H} h")

full_data["Conductivité"], full_data["Conductivité_source"] = cond_iqr, source
full_data["Conductivité_Moyenne_Mobile"] = cond_iqr.rolling("6h", center=True).mean()

graphe([(cond, "avant IQR et lissage", "darkorange"),
        (cond_iqr, "après IQR et lissage", "black")],
       titre="Conductivité", ylab="Conductivité (µS/cm)")

## 12. Température et autres paramètres

Choix automatique, pas de calage sur points de contrôle. Les voies défaillantes ont
déjà été écartées à la cellule des corrections capteur.

In [ ]:
#: {grandeur: [(début, fin, sonde imposée)]} pour sortir du choix automatique.
EXCEPTIONS = {}

for grandeur in ["Température", "Turbidité_(NTU)", "O2_(mg/l)", "Chlorophylle_(RFU)"]:
    print(f"{grandeur} :")
    voies = voies_calees(grandeur)
    full_data[grandeur], full_data[f"{grandeur}_source"] = fusionner(
        voies, choisir_sondes(voies, ORDRE, EXCEPTIONS.get(grandeur, [])))
    print(f"  {full_data[grandeur].notna().sum()} pas  "
          f"{full_data[f'{grandeur}_source'].value_counts().to_dict()}")

## 13. Débit

Deux branches raccordees a H = 21.4 cm, hauteur en metres dans les deux formules.
Le débit est calculé après les corrections du niveau, puis recalculé sur le
niveau interpolé.

In [ ]:
SEUIL_H = 21.4     # cm, raccord des deux branches de la courbe de tarage

def debit(H):
    """Débit en L/s à partir du niveau en cm, jamais négatif."""
    h = pd.to_numeric(H, errors="coerce").astype("float64")
    Q = np.where(h >= SEUIL_H, 8325.3 * (h / 100)**2 - 2266.1 * (h / 100) + 116.21,
                 50.485 * (h / 100) + 1)
    return pd.Series(np.where(h.isna(), np.nan, np.maximum(Q, 0.0)), index=H.index)

full_data["Q_(L/s)"] = debit(full_data["Niveau_(cm)"])
display(full_data[["Niveau_(cm)", "Q_(L/s)"]].describe().round(3))

## 14. Cote NGF, interpolation et statuts

Les lacunes de moins de 12 h sont comblées. `Statut_<grandeur>` dit si la valeur est
mesurée, interpolée ou manquante.

`cote = 311.261 + Niveau_(cm) / 100`. L'ancienne version écrivait `ngf - (ngf - h) / 100`,
qui se simplifie en `308.1484 + h / 100` et plaçait donc le zéro 3.1126 m trop bas.

In [ ]:
NIVEAU_NGF = 311.261       # cote du zéro de l'échelle, None si elle n'est pas connue
MAX_TROU_H = 12

max_pas = int(pd.Timedelta(f"{MAX_TROU_H}h") / pd.Timedelta(PAS))
for col in PARAMETRES:
    origine = full_data[col]
    manquant = origine.isna().to_numpy()
    groupe = np.cumsum(np.r_[True, manquant[1:] != manquant[:-1]])
    tailles = pd.Series(groupe).groupby(groupe).transform("size").to_numpy()
    comble = origine.interpolate(method="time", limit_direction="both")
    comble = comble.mask(manquant & (tailles > max_pas))
    mesures = np.flatnonzero(~manquant)          # pas d'extrapolation hors plage mesurée
    if mesures.size:
        comble.iloc[:mesures[0]] = origine.iloc[:mesures[0]]
        comble.iloc[mesures[-1] + 1:] = origine.iloc[mesures[-1] + 1:]
    full_data[col] = comble
    full_data[f"Statut_{col}"] = np.where(
        ~manquant, "Mesurée", np.where(comble.notna().to_numpy(), "Interpolée", "Manquante"))

if NIVEAU_NGF is not None:
    full_data["Niveau_(mNGF)"] = NIVEAU_NGF + full_data["Niveau_(cm)"] / 100
    full_data["Statut_Niveau_(mNGF)"] = full_data["Statut_Niveau_(cm)"]
    print(f"Zéro de l'échelle à {NIVEAU_NGF:.4f} m NGF")
full_data["Q_(L/s)"] = debit(full_data["Niveau_(cm)"])   # sur le niveau interpolé
full_data["Statut_Q_(L/s)"] = full_data["Statut_Niveau_(cm)"]

display(pd.DataFrame({c: full_data[f"Statut_{c}"].value_counts()
                      for c in PARAMETRES}).fillna(0).astype(int).T)

## 15. Sauvegarde et graphe de synthèse

Un paramètre par grandeur, avec son statut. Le détail capteur par capteur, et la sonde
retenue à chaque pas, restent dans le fichier consolidé écrit à l'assemblage.

In [ ]:
finaux = [c for c in PARAMETRES + ["Q_(L/s)", "Niveau_(mNGF)"] if c in full_data]
colonnes = [c for p in finaux for c in (p, f"Statut_{p}") if c in full_data]
full_data[colonnes].to_excel(SORTIE_FINALE)
print(f"{SORTIE_FINALE} : {len(full_data)} pas x {len(colonnes)} colonnes")

fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True,
                         gridspec_kw={"hspace": 0.05})

axes[0].plot(full_data.index, full_data["Q_(L/s)"].rolling(12, center=True).mean(),
             color="lightseagreen")
axes[0].set_ylabel("Débit (L/s)", color="lightseagreen")
if os.path.exists(PLUIE_PATH):
    pluie = pd.read_csv(PLUIE_PATH)
    ax = axes[0].twinx()
    ax.bar(pd.to_datetime(pluie["Date"], errors="coerce"), pluie["Precipitation (mm)"],
           width=0.8, color="royalblue")
    ax.invert_yaxis()
    ax.set_ylabel("Précipitations (mm)", color="royalblue")

axes[1].plot(full_data.index, full_data["Conductivité_Moyenne_Mobile"], color="black")
axes[1].set_ylabel("Conductivité (µS/cm)")
ax = axes[1].twinx()
ax.plot(full_data.index, full_data["Température"].rolling(12, center=True).mean(), color="crimson")
ax.set_ylabel("Température (°C)", color="crimson")

axes[2].plot(full_data.index, full_data["Turbidité_(NTU)"].rolling(24, center=True).mean(),
             color="darkorange")
axes[2].set_ylabel("Turbidité (NTU)", color="darkorange")
axes[2].set_ylim(0, 100)
ax = axes[2].twinx()
ax.plot(full_data.index, full_data["O2_(mg/l)"].rolling(24, center=True).mean(),
        color="darkmagenta")
ax.set_ylabel("Oxygène (mg/L)", color="darkmagenta")
ax = axes[2].twinx()
ax.spines["right"].set_position(("outward", 45))
ax.plot(full_data.index, full_data["Chlorophylle_(RFU)"].rolling(24, center=True).mean(),
        color="green")
ax.set_ylabel("Chlorophylle (RFU)", color="green")

plt.savefig(SORTIE_SVG, format="svg")
plt.show()

### Contrôle des mesures et des interpolations

Un paramètre à la fois, choisi dans le menu du graphe : les pas mesurés en noir, les pas
comblés par interpolation en rouge. Les trous de plus de 12 h restent vides.

In [ ]:
fig = go.Figure()
for p in finaux:
    for nom, couleur in [("Mesurée", "black"), ("Interpolée", "crimson")]:
        m = (full_data[f"Statut_{p}"] == nom).to_numpy()
        fig.add_trace(go.Scattergl(x=full_data.index[m], y=full_data[p][m], mode="markers",
                                   name=nom, marker=dict(color=couleur, size=3),
                                   visible=(p == finaux[0])))
fig.update_layout(
    updatemenus=[dict(buttons=[dict(label=p, method="update",
                                    args=[{"visible": [q == p for q in finaux for _ in (0, 1)]},
                                          {"yaxis.title.text": p}])
                               for p in finaux],
                      x=0, xanchor="left", y=1.18)],
    title="Statut des données", xaxis_title="Date", yaxis_title=finaux[0],
    template="plotly_white", hovermode="x unified")
fig.show()

print(pd.DataFrame({p: full_data[f"Statut_{p}"].value_counts() for p in finaux})
      .fillna(0).astype(int).T.to_string())